# NumPy Interview Questions — the Vectorization Gate

NumPy screens test one thing above all: **do you think in arrays or in loops?** Every drill below is executed, with timings that prove why the vectorized answer is the right one. Rehearse until the array-first solution is your *first* instinct, not your fallback.

## 1. Creation Idioms — the Ones They Expect Verbatim

In [1]:
import numpy as np

print("zeros(3)        :", np.zeros(3))
print("full(3, 7)      :", np.full(3, 7))
print("arange(1, 10, 3):", np.arange(1, 10, 3))
print("linspace(0, 1, 5):", np.linspace(0, 1, 5))
print("eye(2)          :"); print(np.eye(2))
rng = np.random.default_rng(42)
print("randint 2x3     :"); print(rng.integers(0, 10, size=(2, 3)))

# the interview distinction: arange = step-spaced, linspace = count-spaced
# and: np.random.default_rng() is the modern generator — say that, not np.random.seed


zeros(3)        : [0. 0. 0.]
full(3, 7)      : [7 7 7]
arange(1, 10, 3): [1 4 7]
linspace(0, 1, 5): [0.   0.25 0.5  0.75 1.  ]
eye(2)          :
[[1. 0.]
 [0. 1.]]
randint 2x3     :
[[0 7 6]
 [4 4 8]]


## 2. Broadcasting — the Rule, Then the Proof

**Q: Explain NumPy broadcasting.** The answer interviewers want, verbatim:

> Two shapes are compatible when, compared **right-to-left**, each pair of dimensions is either **equal** or **one of them is 1** (or missing). Dimensions of size 1 stretch to match; anything else is an error.

In [2]:
# (3, 4) + (4,)  ->  the (4,) stretches across all 3 rows
M = np.ones((3, 4))
row = np.array([10, 20, 30, 40])
print("(3,4) + (4,) works:"); print(M + row)

# (3, 1) + (1, 4) -> both stretch into (3, 4): the outer-sum pattern
col = np.arange(3).reshape(3, 1)
print("(3,1) + (1,4) ->"); print(col + row.reshape(1, -1))

# the failure case to know cold:
try:
    M + np.array([1, 2, 3])          # (3,4) + (3,) -> trailing 4 vs 3 mismatch
except ValueError as e:
    print("ValueError:", str(e)[:80])
print("fix: reshape to a column ->", (M + np.array([1, 2, 3]).reshape(3, 1)).shape)

(3,4) + (4,) works:
[[11. 21. 31. 41.]
 [11. 21. 31. 41.]
 [11. 21. 31. 41.]]
(3,1) + (1,4) ->
[[10 20 30 40]
 [11 21 31 41]
 [12 22 32 42]]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 
fix: reshape to a column -> (3, 4)


**The failure case to know cold:** `(3, 4) + (3,)` raises because the trailing dims 4 vs 3 mismatch — the fix is reshaping `(3,)` to `(3, 1)`, i.e. making it a *column*. Interviewers probe exactly this.

## 3. Axis Semantics — Where Everyone Hesitates

**Q: What does `axis=0` mean in `sum`?**

`axis=0` **collapses rows** (aggregate down the columns), `axis=1` **collapses columns** (aggregate across each row). The mnemonic that sticks: *axis is the direction that disappears.* Also know `keepdims=True` for shape-preserving reductions in broadcasting tricks.

In [3]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print("sum axis=0 (down rows -> one per column):", M.sum(axis=0))
print("sum axis=1 (across cols -> one per row) :", M.sum(axis=1))
print("mean axis=0 keepdims:", M.mean(axis=0, keepdims=True).shape, "shape preserved for broadcasting")

sum axis=0 (down rows -> one per column): [5 7 9]
sum axis=1 (across cols -> one per row) : [ 6 15]
mean axis=0 keepdims: (1, 3) shape preserved for broadcasting


## 4. Vectorization — the Speed Proof to Quote

**Q: Why is NumPy faster than Python loops?**

Three stacked reasons, in the order to say them: (1) **contiguous typed memory** (one buffer of C doubles, not an array of PyObject pointers), (2) **loop pushdown into compiled C**, (3) **SIMD** instructions operating on chunks at once. Then quote a number you actually measured:

In [4]:
import time

rng = np.random.default_rng(0)
big = rng.random(2_000_000)

t0 = time.perf_counter(); s_loop = 0.0
for x in big:
    s_loop += x * x
t_loop = time.perf_counter() - t0

t0 = time.perf_counter(); s_vec = float((big ** 2).sum())
t_vec = time.perf_counter() - t0

assert abs(s_loop - s_vec) < 1e-3
print(f"python loop : {t_loop*1000:7.1f} ms")
print(f"vectorized  : {t_vec*1000:7.1f} ms   -> {t_loop/t_vec:.0f}x faster on this machine")

python loop :   322.2 ms
vectorized  :     5.2 ms   -> 62x faster on this machine


## 5. Views vs Copies — the Bug That Bites in Interviews

**Q: Does slicing copy?** No — **basic slicing returns a view**; modifying it modifies the parent. Fancy indexing (boolean masks, index arrays) and `.copy()` return copies. This is the #1 source of silent bugs in take-homes ("why did my original array change?").

In [5]:
a = np.arange(10)

view = a[2:5]                 # basic slice -> view
view[0] = 999
print("after view[0]=999, a[2] is:", a[2], " <- parent changed!")

copy = a[2:5].copy()          # explicit copy
copy[0] = -1
print("after copy edit, a[2] is:", a[2], " <- parent safe")

mask = a > 500                # boolean mask -> fancy indexing -> copy
fancy = a[mask]
fancy[0] = 0
print("after fancy edit, a[2] is:", a[2], " <- fancy indexing copies")

print("shares memory?", np.shares_memory(a, view), np.shares_memory(a, fancy))

after view[0]=999, a[2] is: 999  <- parent changed!
after copy edit, a[2] is: 999  <- parent safe
after fancy edit, a[2] is: 999  <- fancy indexing copies
shares memory? True False


## 6. Boolean Masking — the Everyday Workhorse

*Select ages 18–35 from an array, count them, and zero everything else — no loops.* Combined conditions use `&` / `|` with parentheses — same precedence trap as pandas, worth saying unprompted.

In [6]:
ages = rng.integers(10, 70, size=15)
selected = ages[(ages >= 18) & (ages <= 35)]
print("ages     :", ages)
print("selected :", selected, f"({len(selected)} of {len(ages)})")

ages2 = ages.copy()
ages2[(ages2 < 18) | (ages2 > 35)] = 0
print("zeroed   :", ages2)

# np.where: the conditional one-liner (np.select for multi-way)
cats = np.where(ages < 18, "minor", "adult")
print("where    :", cats[:8])

ages     : [19 17 47 27 15 12 56 36 39 58 51 28 13 23 51]
selected : [19 27 28 23] (4 of 15)
zeroed   : [19  0  0 27  0  0  0  0  0  0  0 28  0 23  0]
where    : ['adult' 'minor' 'adult' 'adult' 'minor' 'minor' 'adult' 'adult']


## 7. Reshape, ravel, and -1

**Q: What does `-1` do in reshape?** It means "infer this dimension." Know that `reshape` returns a view when possible (data order unchanged), and `ravel` vs `flatten`: ravel = view (usually), flatten = always copy.

In [7]:
a = np.arange(12)
print(a.reshape(3, 4))
print("reshape(-1, 1) -> column vector, shape", a.reshape(-1, 1).shape)
print("ravel vs flatten share memory with parent:",
      np.shares_memory(a, a.reshape(3, 4).ravel()),
      np.shares_memory(a, a.reshape(3, 4).flatten()))

# transpose is also a view — mutating T mutates the original
T = a.reshape(3, 4).T
print("transpose shares memory:", np.shares_memory(a.reshape(3, 4), T))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
reshape(-1, 1) -> column vector, shape (12, 1)
ravel vs flatten share memory with parent: True False
transpose shares memory: True


## 8. The Aggregation Combo: argmax, cumsum, clip, round

In [8]:
scores = rng.integers(0, 100, size=10)
print("scores :", scores)
print("argmax :", scores.argmax(), "(index, not value — say that)")
print("cumsum :", scores.cumsum())
print("clip   :", scores.clip(30, 70))
print("round  :", (scores / 7).round(1))
print("unique + counts:", np.unique(scores, return_counts=True)[0][:5])

scores : [96  4 58  0 91  0 45 35 44 60]
argmax : 0 (index, not value — say that)
cumsum : [ 96 100 158 158 249 249 294 329 373 433]
clip   : [70 30 58 30 70 30 45 35 44 60]
round  : [13.7  0.6  8.3  0.  13.   0.   6.4  5.   6.3  8.6]
unique + counts: [ 0  4 35 44 45]


## 9. Linear Algebra One-Liners (They'll Ask for These)

In [9]:
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([3.0, 5.0])

print("A @ b          :", A @ b)                       # matmul operator, not np.dot(a, b)
print("solve Ax=b     :", np.linalg.solve(A, b))       # never invert to solve
print("inv(A) @ b     :", np.linalg.inv(A) @ b, "(same, but slower + less stable — mention this)")
print("det            :", np.linalg.det(A).round(3))
print("norm           :", np.linalg.norm(b).round(3))
vals, vecs = np.linalg.eigh(A)
print("eigh (symmetric):", vals.round(3))

A @ b          : [11. 18.]
solve Ax=b     : [0.8 1.4]
inv(A) @ b     : [0.8 1.4] (same, but slower + less stable — mention this)
det            : 5.0
norm           : 5.831
eigh (symmetric): [1.382 3.618]


**The senior line to say unprompted:** *"I'd use `solve`, not `inv` — explicit inversion is both slower and numerically worse, and for least squares there's `lstsq`."*

## 10. NaN-Safe Operations — the `nan` Prefix

**Q: `mean` of an array with NaNs returns NaN — what do you do?** The `np.nan*` family: `nanmean`, `nanstd`, `nansum`. Then mention `np.isnan` for masking and interpolation as the next step up.

In [10]:
x = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
print("mean (naive):", x.mean(), "<- poisoned by NaN")
print("nanmean     :", np.nanmean(x))
print("isnan count :", np.isnan(x).sum())

# linear interpolation of the NaNs — a real interview follow-up
idx = np.arange(len(x))
good = ~np.isnan(x)
x_filled = x.copy()
x_filled[np.isnan(x)] = np.interp(idx[np.isnan(x)], idx[good], x[good])
print("interpolated:", x_filled)

mean (naive): nan <- poisoned by NaN
nanmean     : 3.0
isnan count : 2
interpolated: [1. 2. 3. 4. 5.]


## 11. The One-Line Challenge Set

Five tasks, five one-liners. These are the exact phrasings screens use.

In [11]:
m = rng.integers(-5, 6, size=(4, 5))

# 1) row-wise standardized (z-scores)
z = (m - m.mean(axis=1, keepdims=True)) / m.std(axis=1, keepdims=True)
assert z.shape == m.shape and abs(z.mean(axis=1)).max() < 1e-9

# 2) replace negatives with 0 (the ReLU answer)
relu = np.maximum(m, 0)

# 3) softmax of a vector, numerically stable
v = rng.normal(size=6)
e = np.exp(v - v.max()); sm = e / e.sum()
assert abs(sm.sum() - 1) < 1e-9

# 4) pairwise squared distances between two point sets (X:(n,d), Y:(m,d))
X = rng.normal(size=(5, 3)); Y = rng.normal(size=(4, 3))
D = ((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
assert D.shape == (5, 4)

# 5) one-hot encode a label vector
labels = np.array([0, 2, 1, 2])
onehot = (labels[:, None] == np.arange(3)).astype(int)

print("all five one-liners verified: shapes", z.shape, relu.shape, sm.shape, D.shape, onehot.shape)
print("one-hot:"); print(onehot)

all five one-liners verified: shapes (4, 5) (4, 5) (6,) (5, 4) (4, 3)
one-hot:
[[1 0 0]
 [0 0 1]
 [0 1 0]
 [0 0 1]]


## 12. dtype Awareness — the Silent Precision Trap

**Q: What's wrong with `np.array([1, 2, 3]) / 2`?** Nothing in Python 3 (true division → float64). But `np.array([1,2,3]) // 2` stays int — and integer *overflow* wraps silently. dtype discipline is what interviewers look for in senior candidates:

In [12]:
ints = np.array([100, 50, 99], dtype=np.int8)    # int8 max = 127
print("int8 wraparound:", ints + 100, " <- values past 127 wrap negative, silently")

f16 = np.array([30000.0], dtype=np.float16)
print("float16 precision:", f16, "->", f16 + 1, "(1 vanishes in half precision)")

print("safe: cast explicitly ->", ints.astype(np.int32) + 100)
print("the rule: state dtype when it matters; never let it be an accident")

int8 wraparound: [ -56 -106  -57]  <- values past 127 wrap negative, silently
float16 precision: [30000.] -> [30000.] (1 vanishes in half precision)
safe: cast explicitly -> [200 150 199]
the rule: state dtype when it matters; never let it be an accident


## Rapid-Fire Table — Say These Without Pausing

| Question | Answer |
| --- | --- |
| Difference `np.dot`, `@`, `np.matmul`? | For 2-D, identical; `@`/`matmul` broadcast matrix stacks, `dot` treats N-D as stacks of dot products — prefer `@` |
| What's a structured array? | NumPy's record type — array of tuples with named, typed fields; the precursor to pandas |
| `arange` vs `linspace`? | Step-defined vs count-defined; floating `arange` can include the endpoint by rounding error — use `linspace` for floats |
| How to read a CSV fast? | `np.loadtxt`/`genfromtxt` (slow, for small files); for real work `pandas.read_csv` then `.to_numpy()` |
| Random numbers — old or new API? | `default_rng()`; the legacy `np.random.seed`/`np.random.rand` is frozen legacy — say you use the new generator |
| `np.nan` == `np.nan`? | **False** — NaN compares unequal to everything; test with `np.isnan`, never `== nan` |
| Memory layout? | C order (row-major) default; `order='F'` column-major; mismatched order in big loops kills cache performance |
| What is `np.einsum`? | Index-notation contraction — expresses dot/matmul/batch products in one call; the "I read the docs" flex |